# 03 — Five-model baseline comparison with GroupKFold

This notebook trains at least five model families on **training trips only**. `GroupKFold` keeps all candidate rows from a trip in one fold, preventing candidate rows from the same trip appearing in both training and validation folds.

The five required baseline candidates are Logistic Regression, KNN, Decision Tree, Random Forest and HistGradientBoosting. The selection criterion is mean cross-validated macro-F1, considered alongside log loss, accuracy, class metrics and runtime. Student-testing data is not used in this baseline model-selection step.

In [ ]:
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss

SEED = 42
TRAIN_CSV = Path("data/processed/train_candidates.csv")
MANIFEST = Path("configs/person_b_feature_manifest.json")
OUTPUT = Path("experiments/results/person_b")
OUTPUT.mkdir(parents=True, exist_ok=True)
if not TRAIN_CSV.exists() or not MANIFEST.exists():
    raise FileNotFoundError("Run 02_candidate_table_validation.ipynb after Person A supplies candidate CSVs.")
train_df = pd.read_csv(TRAIN_CSV)
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
FEATURES = manifest["feature_columns"]
TARGET = manifest["target_column"]
X, y, groups = train_df[FEATURES], train_df[TARGET].astype(int), train_df["trip_id"]
if y.nunique() != 2:
    raise ValueError("Training target must contain both classes")
n_splits = min(5, groups.nunique())
if n_splits < 2:
    raise ValueError("Need at least two distinct training trips")
cv = GroupKFold(n_splits=n_splits)
print(f"Rows={len(train_df):,}; trips={groups.nunique()}; features={len(FEATURES)}; folds={n_splits}")

In [ ]:
def build_models(seed=SEED):
    return {
        "logistic_regression": make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed)
        ),
        "knn": make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(),
            KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=-1)
        ),
        "decision_tree": make_pipeline(
            SimpleImputer(strategy="median"),
            DecisionTreeClassifier(max_depth=14, min_samples_leaf=5, class_weight="balanced", random_state=seed)
        ),
        "random_forest": make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestClassifier(n_estimators=250, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1, random_state=seed)
        ),
        "hist_gradient_boosting": make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingClassifier(max_iter=180, learning_rate=0.08, max_leaf_nodes=31, random_state=seed)
        ),
    }

models = build_models()
print("Models:", list(models))

In [ ]:
fold_rows = []
for model_name, model in models.items():
    print("Running", model_name)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y, groups), start=1):
        estimator = clone(model)
        start = time.perf_counter()
        estimator.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        train_seconds = time.perf_counter() - start
        start = time.perf_counter()
        pred = estimator.predict(X.iloc[va_idx])
        inference_seconds = time.perf_counter() - start
        proba = estimator.predict_proba(X.iloc[va_idx])[:, 1]
        fold_rows.append({
            "model": model_name, "fold": fold,
            "accuracy": accuracy_score(y.iloc[va_idx], pred),
            "precision": precision_score(y.iloc[va_idx], pred, zero_division=0),
            "recall": recall_score(y.iloc[va_idx], pred, zero_division=0),
            "macro_f1": f1_score(y.iloc[va_idx], pred, average="macro", zero_division=0),
            "log_loss": log_loss(y.iloc[va_idx], proba, labels=[0, 1]),
            "train_seconds": train_seconds,
            "inference_seconds": inference_seconds,
            "validation_rows": len(va_idx),
            "validation_trips": groups.iloc[va_idx].nunique(),
        })
fold_results = pd.DataFrame(fold_rows)
fold_results.to_csv(OUTPUT / "baseline_groupkfold_folds.csv", index=False)
summary = fold_results.groupby("model").agg(
    accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
    precision_mean=("precision", "mean"), recall_mean=("recall", "mean"),
    macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
    log_loss_mean=("log_loss", "mean"), train_seconds_mean=("train_seconds", "mean"),
    inference_seconds_mean=("inference_seconds", "mean")
).sort_values("macro_f1_mean", ascending=False)
summary.to_csv(OUTPUT / "baseline_model_comparison.csv")
display(summary)
print("Saved fold-level and summary results to", OUTPUT)

In [ ]:
import matplotlib.pyplot as plt
ax = summary["macro_f1_mean"].sort_values().plot(kind="barh", figsize=(9, 5))
ax.set_xlabel("Mean GroupKFold macro-F1")
ax.set_title("Baseline model comparison (training trips only)")
ax.grid(axis="x", alpha=.25)
plt.tight_layout(); plt.show()

best_baseline = summary.index[0]
print("Best untuned baseline by mean macro-F1:", best_baseline)
print("This is a development result, not the final model choice. Continue with Bayesian optimisation and student-test evaluation.")

## Interpretation notes

- Macro-F1 gives equal weight to the positive and negative classes; it is useful when candidate labels are imbalanced.
- Log loss evaluates probability quality, which matters because Person C may use classifier probabilities as HMM emission scores.
- GroupKFold splits by `trip_id`, not by candidate row.
- KNN may be slow or memory-intensive on a large candidate table; record runtime rather than silently skipping it.
- The final choice must consider student-test performance and practical inference time, not training accuracy alone.